# Retail Inventory AI — Detection Checkpoint (Week 3)

## YOLOv8m on SKU-110K — full detection pipeline & Before/After comparison

_Project: Intelligent Retail Product Detection & Inventory Management_
_Date: 2026-06-07_

This notebook documents the **entire Module-1 detection pipeline** end to end, and proves the
checkpoint with a **before vs. after** comparison on the same shelf images.

It is organised in three parts:

| Part | Contents | Runs here? |
|------|----------|------------|
| **1. Data pipeline**   | How SKU-110K was downloaded and preprocessed (CSV → YOLO format) | 📄 **Documentation only** |
| **2. Training**        | The full GCP automation: launch the L4 VM → train → validate → sync → self-delete | 📄 **Documentation only** |
| **3. Before / After**  | COCO-pretrained vs. our fine-tuned `best.pt` on identical images + metrics | ▶️ **Runs** |

> **Why parts 1–2 do not run here:** the dataset is ~13 GB and training takes ~9.5 h on an
> L4 GPU — reproducing them in Colab would be slow, expensive, and pointless. Those steps
> already ran on GCP. Below we *show the exact code we used* so the pipeline is fully
> transparent and reproducible, then **execute only the lightweight inference** needed to
> demonstrate the result.

**Detection targets:** mAP@0.5 ≥ 0.70 · Precision ≥ 0.75 · Recall ≥ 0.70.

**Achieved:**

| Variant   | mAP@0.5    | mAP@0.5:0.95 | Precision  | Recall    | Time cap |
|-----------|------------|--------------|------------|-----------|----------|
| YOLOv8m   | **0.9209** | 0.5937       | **0.9193** | **0.8824**| 9.5 h    |
| YOLOv11m  | _pending_  | _pending_    | _pending_  | _pending_ | 8.0 h    |

> **How to run (personal Google account, no corp auth):**
> 1. Open this notebook in Colab (File → Upload notebook, or File → Open → GitHub).
> 2. Grab the per-variant artifacts from this repo at `detection/artifacts/v8/` and
>    `detection/artifacts/v11/` — six files total, renamed before upload to:
>    `best_v8.pt`, `metrics_v8.json`, `results_v8.png`,
>    `best_v11.pt`, `metrics_v11.json`, `results_v11.png`.
> 3. Runtime → **Run all**. When §3.3 prompts, multi-select all six.
>
> Only Part 3 executes (~2–3 minutes on CPU for 10 sample images × 2 fine-tuned
> models); Parts 1–2 are read-only documentation. No GCS, no `gcloud`, no corporate
> sign-in anywhere.

---
# Part 1 — Data Pipeline  📄 *(documentation, not executed)*

## 1.1  Dataset: SKU-110K

SKU-110K is a dense retail-shelf detection dataset from Goldman et al., CVPR 2019.

| Property             | Value                                              |
|----------------------|----------------------------------------------------|
| Images               | ~11,762 (train / val / test)                       |
| Annotations          | ~1.7 M bounding boxes                              |
| Avg objects / image  | ~147                                               |
| Classes              | 1 (`object` — localization only)                   |
| Source               | retail store shelves worldwide                     |
| License              | CC BY-NC 4.0 (research use)                        |

Why we chose it: it is the largest publicly-available **dense** retail dataset, with hundreds of
small objects per image — exactly the regime where stock COCO models fail and a fine-tuned
detector earns its keep.

## 1.2  How we downloaded it

We used the **Ultralytics built-in dataset spec**, which downloads *and* converts SKU-110K
automatically the first time it is referenced — no Kaggle account or manual step needed. Under
the hood it fetches the official `SKU110K_fixed.tar.gz` (~13 GB) and runs the CSV → YOLO
conversion described in 1.3.

**The one line that triggers the download** (inside the training command — see Part 2):

In [ ]:
# `data=SKU-110K.yaml` is an Ultralytics-bundled spec. On first use it:
#   1. downloads http://trax-geometry.s3.amazonaws.com/cvpr_challenge/SKU110K_fixed.tar.gz (~13 GB)
#   2. unpacks it to  /datasets/SKU-110K/{images,labels}/{train,val,test}
#   3. converts the CSV annotations to YOLO-format .txt label files (see 1.3)
#
# !yolo detect train model=yolov8m.pt data=SKU-110K.yaml ...


**Equivalent explicit/manual download** (what the spec does for you), shown for full
transparency in case the auto-spec is ever unavailable:

In [ ]:
# Manual alternative — fetch and unpack the raw dataset yourself.
# DOCUMENTATION ONLY — do not run here (~13 GB download).
#
# !mkdir -p /datasets && cd /datasets \
#   && wget http://trax-geometry.s3.amazonaws.com/cvpr_challenge/SKU110K_fixed.tar.gz \
#   && tar -xzf SKU110K_fixed.tar.gz
#
# Result tree:
#   /datasets/SKU110K_fixed/
#     ├── images/       # ~11,762 JPGs
#     └── annotations/
#         ├── annotations_train.csv
#         ├── annotations_val.csv
#         └── annotations_test.csv


## 1.3  Preprocessing — CSV → YOLO label format

SKU-110K ships annotations as **CSV**, one row per box:

```
image_name, x1, y1, x2, y2, class, image_width, image_height
test_0.jpg, 12,  34, 120, 210, object, 1920, 1080
test_0.jpg, 145, 30, 260, 215, object, 1920, 1080
...
```

YOLO needs **one `.txt` per image**, one row per box, with **normalised** `class cx cy w h`
(centre-x, centre-y, width, height — each in `[0, 1]`):

```
0  0.034375  0.112963  0.056250  0.162963
0  0.105729  0.113426  0.059896  0.171296
...
```

The conversion below mirrors what the Ultralytics `SKU-110K.yaml` spec runs automatically —
shown explicitly so a reviewer can see exactly what was applied to every image.

In [ ]:
# preprocess_sku110k.py  — CSV bounding boxes -> normalised YOLO .txt labels
# DOCUMENTATION ONLY — already executed by the Ultralytics spec on the GCP VM.

from pathlib import Path
import pandas as pd
from tqdm import tqdm

DATA = Path('/datasets/SKU110K_fixed')
COLS = ['image', 'x1', 'y1', 'x2', 'y2', 'class', 'img_w', 'img_h']

for split in ['train', 'val', 'test']:
    # Read the CSV — no header, columns in fixed order.
    df = pd.read_csv(DATA / 'annotations' / f'annotations_{split}.csv', names=COLS)

    out_dir = DATA / 'labels' / split
    out_dir.mkdir(parents=True, exist_ok=True)

    # All boxes belonging to the same image -> one .txt file.
    for image_name, g in tqdm(df.groupby('image'), desc=f'{split} labels'):
        lines = []
        for _, r in g.iterrows():
            # Convert pixel-space (x1,y1,x2,y2) to normalised YOLO (cx,cy,w,h).
            w  = (r.x2 - r.x1) / r.img_w           # width      (normalised)
            h  = (r.y2 - r.y1) / r.img_h           # height     (normalised)
            cx = (r.x1 + r.x2) / 2 / r.img_w       # centre-x   (normalised)
            cy = (r.y1 + r.y2) / 2 / r.img_h       # centre-y   (normalised)
            # Class id 0 = 'object' (single-class dataset).
            lines.append(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')

        # One label file per image, e.g. labels/train/train_0.txt
        (out_dir / f'{Path(image_name).stem}.txt').write_text('\n'.join(lines))


**Edge cases handled by Ultralytics during its label scan:**

- A handful of SKU-110K images have **corrupt/zero-area boxes** — Ultralytics skips them
  automatically. Our run dropped exactly **3 corrupt training images** (logged in `train.log`).
- A handful of JPEGs have **truncated EOI markers** ("Corrupt JPEG data: extraneous bytes…");
  these are auto-restored and saved back, so training continues unaffected.

## 1.4  The dataset spec — `SKU-110K.yaml`

The YOLO data spec ties images + labels together and declares the single class. Ultralytics
bundles this file; reproduced here for completeness:

In [ ]:
# SKU-110K.yaml  (Ultralytics-bundled — shown for transparency)
# DOCUMENTATION ONLY — Ultralytics writes this on first use.

yaml_spec = '''
path: ../datasets/SKU-110K       # dataset root
train: train.txt                 # list of training image paths
val:   val.txt                   # list of validation image paths
test:  test.txt                  # list of test image paths

names:
  0: object                      # single class — detection / localization only

# `download:` field holds the python that fetches the tar.gz and runs the
# CSV -> YOLO conversion from 1.3 the first time the spec is used.
'''
print(yaml_spec)


## 1.5  Image preprocessing at train/inference time

These transforms are applied **on-the-fly** by YOLO (not as a separate offline step):

| Step                    | Train split            | Val / Test split    |
|-------------------------|------------------------|---------------------|
| Letterbox resize        | 1280 × 1280            | 1280 × 1280         |
| Pixel normalisation     | divide by 255 → [0, 1] | same                |
| Mosaic augmentation     | ✓                      | ✗                   |
| HSV jitter              | ✓                      | ✗                   |
| Horizontal flip         | ✓                      | ✗                   |
| Random scale + translate| ✓                      | ✗                   |

**Why imgsz = 1280 and not the YOLO default 640?** SKU-110K averages ~147 tiny products per image.
At 640 px, individual SKUs collapse to a few pixels and recall craters. 1280 px keeps them
recognisable while still fitting on the L4's 24 GB VRAM (with AutoBatch landing on batch=3).

---
# Part 2 — Training Pipeline  📄 *(the exact GCP automation, not executed here)*

We fine-tune **YOLO** (COCO-pretrained) on SKU-110K on a single **NVIDIA L4** GPU
(`g2-standard-8`, 24 GB VRAM). The same launcher trains both v8 and v11 — variant
selected via env vars (`MODEL`, `TIME_HOURS`, `VARIANT`) so a single set of scripts
covers every run.

```
  (laptop)                              (GCP)
  MODEL=yolov8m.pt  TIME_HOURS=9.5  VARIANT=v8   bash launch_vm.sh
  MODEL=yolo11m.pt  TIME_HOURS=8.0  VARIANT=v11  bash launch_vm.sh
                                  │
                                  ▼  L4 VM boots (NO public IP — egress via Cloud NAT)
                       train_sku110k.sh runs as root startup
                                  │
                                  ├─ install libs + ultralytics (via NAT)
                                  ├─ yolo detect train  (≤ TIME_HOURS cap)
                                  ├─ yolo detect val
                                  ├─ extract metrics.json
                                  ├─ rsync /opt/runs -> gs://.../results/<VARIANT>/
                                  └─ delete self                     ← cost safety
```

The two scripts — `detection/launch_vm.sh` and `detection/train_sku110k.sh` — are split into
the logical blocks below.

## 2.1  Launcher — create the L4 VM (`detection/launch_vm.sh`)

Runs **on the laptop** (not on GCP). It creates the VM, attaches an L4, picks the Deep-Learning
PyTorch image (CUDA + driver preinstalled), and uploads the training script as the boot
**startup-script** so training begins immediately on first boot — no SSH required.

### 2.1.1  Inputs and defaults

In [ ]:
# detection/launch_vm.sh  (top of file)
# DOCUMENTATION ONLY — already ran on the laptop.

# !/bin/bash
# set -euo pipefail
#
# # Required: pick the GCP project + bucket. Zone defaults to us-central1-a (L4 quota lives there).
# : "${PROJECT_ID:?set PROJECT_ID (e.g. export PROJECT_ID=my-gcp-project)}"
# ZONE="${ZONE:-us-central1-a}"
# REGION="${ZONE%-*}"
# BUCKET="${BUCKET:-gs://${PROJECT_ID}-sku110k-yolo}"
#
# # Variant knobs — the same launcher trains v8, v11, ...:
# MODEL="${MODEL:-yolov8m.pt}"          # or yolo11m.pt, ...
# TIME_HOURS="${TIME_HOURS:-9.5}"       # hard wall-clock cap (the v11 run uses 8.0)
# VARIANT="${VARIANT:-v8}"              # subdir under $BUCKET/results/
# INSTANCE="${INSTANCE:-sku110k-train-${VARIANT}}"


### 2.1.2  Subnet egress precheck — required because the VM has no public IP

Corporate policy: **VMs must not have an external/public IP**. Without one, every outbound
network call (apt-get, pip, the `yolo*.pt` CDN download, the SKU-110K dataset, `gsutil`,
the self-delete `gcloud`) must route through **Cloud NAT** on the subnet. Private Google
Access alone is not enough — it covers GCS and Google APIs but not Ubuntu repos or PyPI.
The launcher aborts with an actionable error if NAT is missing.

In [ ]:
# # --- Precheck: subnet must have Cloud NAT --------------------------
# ROUTER="$(gcloud compute routers list --regions="$REGION" --format='value(name)' | head -1)"
# NAT_FOUND=""
# if [[ -n "$ROUTER" ]]; then
#   NAT_FOUND="$(gcloud compute routers describe "$ROUTER" --region="$REGION" \
#                  --format='value(nats[].name)')"
# fi
# if [[ -z "$NAT_FOUND" ]]; then
#   echo "ERROR: no Cloud NAT in $REGION. The no-public-IP VM will hang on apt/pip/yolo*.pt."
#   echo "Enable with: gcloud compute routers create nat-router-$REGION --region=$REGION \\"
#   echo "             --network=default && gcloud compute routers nats create nat-config \\"
#   echo "             --router=nat-router-$REGION --region=$REGION \\"
#   echo "             --auto-allocate-nat-external-ips --nat-all-subnet-ip-ranges"
#   exit 1
# fi


### 2.1.3  Inject placeholders into the startup script

The training script lives in the repo with **four** placeholders. The launcher renders
them before uploading — keeps bucket/model/time/variant out of the committed script:
`__BUCKET__`, `__MODEL__`, `__TIME_HOURS__`, `__VARIANT__`.

In [ ]:
# # Resolve this script's directory so it works from anywhere.
# SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
#
# STARTUP="$(mktemp)"
# sed -e "s|__BUCKET__|${BUCKET}|g" \
#     -e "s|__MODEL__|${MODEL}|g" \
#     -e "s|__TIME_HOURS__|${TIME_HOURS}|g" \
#     -e "s|__VARIANT__|${VARIANT}|g" \
#     "$SCRIPT_DIR/train_sku110k.sh" > "$STARTUP"


### 2.1.4  Create the VM — security-policy-compliant flags

`--no-address` is the policy-compliance flag — it strips the external IP entirely.
`--subnet=default` and `--service-account=…-compute@…` are surfaced (rather than
implicit) so the bucket-IAM binding the SA needs is obvious.

In [ ]:
# # Default service account: project's compute SA (already has roles/storage.objectAdmin).
# PROJECT_NUMBER="$(gcloud projects describe "$PROJECT_ID" --format='value(projectNumber)')"
# SERVICE_ACCOUNT="${SERVICE_ACCOUNT:-${PROJECT_NUMBER}-compute@developer.gserviceaccount.com}"
#
# gcloud compute instances create "$INSTANCE" \
#   --project="$PROJECT_ID" \
#   --zone="$ZONE" \
#   --subnet=default \                              # explicit (was implicit)
#   --no-address \                                  # NO public IP — egress via Cloud NAT
#   --service-account="$SERVICE_ACCOUNT" \          # surfaces the SA we IAM-bound to the bucket
#   --machine-type=g2-standard-8 \                  # 8 vCPU, 32 GB RAM (L4 host shape)
#   --accelerator=type=nvidia-l4,count=1 \          # 1 × NVIDIA L4 (24 GB VRAM)
#   --maintenance-policy=TERMINATE \                # GPUs require TERMINATE, not MIGRATE
#   --restart-on-failure \
#   --image-family=pytorch-2-9-cu129-ubuntu-2204-nvidia-580 \
#   --image-project=deeplearning-platform-release \
#   --boot-disk-size=200GB \
#   --boot-disk-type=pd-ssd \
#   --metadata="install-nvidia-driver=True" \
#   --metadata-from-file=startup-script="$STARTUP" \
#   --scopes=https://www.googleapis.com/auth/cloud-platform   # GCS write + self-delete


### 2.1.5  IAM gotcha — bucket access for the VM

The VM uses the project's default Compute Engine service account. That SA needs **write access
to the results bucket** so the training script can `rsync` artifacts back. Granted **once per
project**:

In [ ]:
# One-time bucket binding so the default compute SA can write results back.
# Run from the laptop (not on the VM).
#
# !gcloud storage buckets add-iam-policy-binding gs://${PROJECT_ID}-sku110k-yolo \
#   --member=serviceAccount:$(gcloud projects describe ${PROJECT_ID} --format='value(projectNumber)')-compute@developer.gserviceaccount.com \
#   --role=roles/storage.objectAdmin
#
# (We learned the hard way — the first run trained successfully but the EXIT trap's
# rsync silently failed with a 403 and the VM kept running idle. Now baked into setup.)


## 2.2  Training script — `detection/train_sku110k.sh`

Runs **on the VM** as root, automatically, on first boot. It owns the entire lifecycle:
install → train → validate → sync → self-delete.

### 2.2.1  Setup + the cost-safety EXIT trap

The trap is the most important block: it guarantees the VM syncs its results to GCS and
deletes itself **on every exit path** — success, failure, or crash. Without it, a silent
crash leaves a $1/hr GPU running indefinitely.

In [ ]:
# !/bin/bash
# set -uo pipefail
#
# # All four placeholders are sed-replaced by launch_vm.sh before this is uploaded.
# BUCKET="__BUCKET__"        # gs://...-sku110k-yolo
# MODEL="__MODEL__"          # yolov8m.pt or yolo11m.pt
# TIME_HOURS="__TIME_HOURS__"  # 9.5 (v8) or 8.0 (v11) — hard wall-clock cap
# VARIANT="__VARIANT__"      # v8 / v11 — subdir under $BUCKET/results/
#
# RUN_DIR=/opt/runs
# LOG="$RUN_DIR/train.log"
# mkdir -p "$RUN_DIR"
# : > "$LOG"                 # truncate log
#
# # ---- COST-SAFETY EXIT TRAP --------------------------------------------------
# # Fires no matter how the script exits — success, error, even SIGTERM.
# cleanup() {
#   rc=$?
#   echo "=== cleanup (exit $rc): syncing /opt/runs to GCS, then self-deleting ==="
#   # Per-variant subpath so v8 and v11 runs never overwrite each other.
#   gsutil -m rsync -r "$RUN_DIR" "$BUCKET/results/$VARIANT/" || echo "WARN: gsutil sync failed"
#
#   # Read our own VM name + zone from the metadata server (no hard-coding).
#   NAME=$(curl -s -H "Metadata-Flavor: Google" \
#     http://metadata.google.internal/computeMetadata/v1/instance/name)
#   ZONE=$(curl -s -H "Metadata-Flavor: Google" \
#     http://metadata.google.internal/computeMetadata/v1/instance/zone | awk -F/ '{print $NF}')
#
#   echo "=== deleting self: $NAME in $ZONE ==="
#   gcloud compute instances delete "$NAME" --zone "$ZONE" --quiet
# }
# trap cleanup EXIT


### 2.2.2  Logging — write to a file, NOT through the metadata runner

Subtle but critical bug we hit and fixed: YOLO's progress bars use `\r` (no newline) and produce
multi-KB "lines". GCP's metadata script-runner pipes stdout through Go's `bufio.Scanner` with a
**64 KB token limit** → "token too long" → the runner kills the child process mid-training.

**Fix:** redirect all heavy output directly to a file (`>> "$LOG" 2>&1`) and only echo short
milestone messages to stdout. The scanner never sees the long lines.

In [ ]:
# # Wait for the NVIDIA driver to come up (DLVM preinstalls it; this is just a safety guard).
# until nvidia-smi >> "$LOG" 2>&1; do echo "waiting for GPU driver..."; sleep 15; done
# echo "GPU ready."
#
# # OpenCV (an Ultralytics dep) needs system OpenGL libs not in the base image.
# export DEBIAN_FRONTEND=noninteractive
# echo "installing system libs (libgl1, libglib2.0-0)..."
# apt-get update                               >> "$LOG" 2>&1 \
#   && apt-get install -y libgl1 libglib2.0-0  >> "$LOG" 2>&1
#
# echo "installing ultralytics..."
# pip install --upgrade "ultralytics>=8.2"     >> "$LOG" 2>&1
#
# # Less spammy progress bars — every 10s instead of every 100ms.
# export TQDM_MININTERVAL=10
# export PYTHONUNBUFFERED=1


### 2.2.3  Train — the core command

This is the **single command that produced our model**. Every flag below was deliberately
chosen for the L4 + SKU-110K combination.

In [ ]:
# echo "=== starting $MODEL training (output -> $LOG) ==="
#
# yolo detect train \
#   model="$MODEL"      \   # yolov8m.pt OR yolo11m.pt (COCO-pretrained, transfer learning)
#   data=SKU-110K.yaml  \   # auto-downloads + converts the dataset (Part 1)
#   epochs=50           \   # UPPER BOUND only — the time cap below usually stops first
#   time="$TIME_HOURS"  \   # HARD wall-clock cap: 9.5h for v8, 8.0h for v11 (~$1.25 cheaper)
#   imgsz=1280          \   # high resolution for tiny dense products (see 1.5)
#   batch=-1            \   # AutoBatch: pick largest batch fitting ~60% VRAM (chose 3)
#   cos_lr=True         \   # cosine learning-rate schedule
#   project=/opt/runs   \   # output dir
#   name=sku110k        \
#   exist_ok=True       \
#   >> "$LOG" 2>&1
#
# echo "=== training finished (exit $?) ==="


**Why `time=9.5` instead of a fixed epoch count?**
In Ultralytics, `time=` *overrides* `epochs` and auto-scales the number of epochs to consume
the entire budget — trading wall-clock for accuracy. On the L4 we got further into the schedule
(epoch 47 / 50) than we would have on a slower GPU in the same window. It also makes the run
**cost-deterministic**: ~9.5 h × ~$0.83/h ≈ **$8** per training run, no surprises.

### 2.2.4  Validate — produce the target metrics

In [ ]:
# echo "=== running final validation ==="
#
# yolo detect val \
#   model="$RUN_DIR/sku110k/weights/best.pt" \   # the best epoch's checkpoint
#   data=SKU-110K.yaml \
#   imgsz=1280 \                                  # same resolution as training
#   project="$RUN_DIR" \
#   name=sku110k_val \
#   exist_ok=True \
#   >> "$LOG" 2>&1


### 2.2.5  Extract the four target metrics into `metrics.json`

Parses the per-epoch CSV that Ultralytics writes and pulls out the final-epoch row. Wrapped in
`|| true` so a parse hiccup can never skip the cleanup trap.

In [ ]:
# python3 - "$RUN_DIR" >> "$LOG" 2>&1 <<'PY' || true
# import csv, json, os, sys
#
# rd = sys.argv[1]
# csvf = os.path.join(rd, "sku110k", "results.csv")
#
# # Some early rows have empty mAP50 cells (warmup); skip them.
# rows = [r for r in csv.DictReader(open(csvf))
#         if r.get("metrics/mAP50(B)", "").strip()]
# last = rows[-1]
#
# out = {
#     "mAP_50":    round(float(last["metrics/mAP50(B)"]), 4),
#     "mAP_50_95": round(float(last["metrics/mAP50-95(B)"]), 4),
#     "precision": round(float(last["metrics/precision(B)"]), 4),
#     "recall":    round(float(last["metrics/recall(B)"]), 4),
#     "epoch":     int(float(last["epoch"])),
# }
# json.dump(out, open(os.path.join(rd, "metrics.json"), "w"), indent=2)
# print(json.dumps(out, indent=2))
# PY
#
# echo "=== training + validation complete; metrics.json written ==="
# # Script ends here -> EXIT trap fires -> sync to GCS + delete VM.


### 2.2.6  What lands in GCS

Each run lands in its own variant subdirectory so v8 and v11 never collide:

```
gs://ehc-mgrandhi-bc801a-sku110k-yolo/results/
├── v8/                         ← YOLOv8m,  9.5 h cap   (already done)
│   ├── best.pt    last.pt     ← fine-tuned + last-epoch checkpoints (~50 MiB ea)
│   ├── metrics.json            ← four target numbers
│   ├── train.log               ← full training log     (~20 MiB)
│   ├── sku110k/                ← training run dir
│   │   ├── results.png         ← loss + mAP curves
│   │   ├── results.csv         ← per-epoch metrics
│   │   ├── args.yaml           ← reproducibility record
│   │   ├── confusion_matrix.png
│   │   ├── BoxP_curve.png  BoxR_curve.png  BoxF1_curve.png  BoxPR_curve.png
│   │   ├── train_batch*.jpg    ← training-time augmentations preview
│   │   ├── val_batch*_pred.jpg ← validation predictions
│   │   └── weights/{best,last}.pt
│   └── sku110k_val/            ← final val pass viz
└── v11/                        ← YOLOv11m, 8.0 h cap   (next run)
    └── (same layout)
```

`best.pt`, `metrics.json`, `results.png` from each run are **copied into this repo** at
`detection/artifacts/v8/` and `detection/artifacts/v11/` so Part 3 below can be reproduced
from a personal Google account with no GCS access at all.

---
# Part 3 — Before vs. After  ▶️ *(this part runs)*

Everything below executes — and **runs entirely on a personal Google account**. No corporate
auth, no GCS, no `gcloud`. The artifacts live in this repo at `detection/artifacts/v8/` and
`detection/artifacts/v11/`; you upload them into the Colab runtime with one click.

## 3.1  Setup

In [ ]:
# Install Ultralytics (YOLO v8 + v11 share the same package). Quiet to keep the log clean.
!pip install -q "ultralytics>=8.2"

import os, glob, json
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Confirm GPU if present (inference works fine on CPU too).
!nvidia-smi -L || echo 'No GPU - running inference on CPU (fine for a few images).'

## 3.2  Configuration

In [ ]:
# Display + inference knobs.
CONF      = 0.25       # confidence threshold for display
IMGSZ     = 1280       # match training resolution (dense small objects)
N_SAMPLES = 10         # how many shelf images to compare

## 3.3  Upload the per-variant artifacts (one-time per Colab session)

Grab six files from this repo and **rename each one with the variant suffix below before
upload** — the notebook keys off these exact filenames:

| Source path in repo                       | Rename to (upload as)  | Size      |
|-------------------------------------------|------------------------|-----------|
| `detection/artifacts/v8/best.pt`          | `best_v8.pt`           | ~50 MiB   |
| `detection/artifacts/v8/metrics.json`     | `metrics_v8.json`      | ~100 B    |
| `detection/artifacts/v8/results.png`      | `results_v8.png`       | ~300 KiB  |
| `detection/artifacts/v11/best.pt`         | `best_v11.pt`          | ~50 MiB   |
| `detection/artifacts/v11/metrics.json`    | `metrics_v11.json`     | ~100 B    |
| `detection/artifacts/v11/results.png`     | `results_v11.png`      | ~300 KiB  |

Multi-select all six in the file picker. **At least one variant** (v8 or v11) must be fully
present — partial uploads are fine if you only have one variant trained so far.

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

def variant_complete(v):
    return all(f'{kind}_{v}.{ext}' in uploaded
               for kind, ext in [('best','pt'), ('metrics','json'), ('results','png')])

HAVE_V8  = variant_complete('v8')
HAVE_V11 = variant_complete('v11')
print(f'v8 artifacts complete:  {HAVE_V8}')
print(f'v11 artifacts complete: {HAVE_V11}')
assert HAVE_V8 or HAVE_V11, 'Need at least one full variant (best_*.pt + metrics_*.json + results_*.png).'

## 3.4  Load up to three models

- **BEFORE** — stock COCO-pretrained YOLOv8m (auto-download from public Ultralytics CDN).
- **AFTER v8** — our fine-tuned `best_v8.pt` (loaded only if v8 was uploaded).
- **AFTER v11** — our fine-tuned `best_v11.pt` (loaded only if v11 was uploaded).

In [ ]:
before_model = YOLO('yolov8m.pt')
print('Loaded BEFORE model: yolov8m.pt (COCO-pretrained)')

v8_model  = YOLO('best_v8.pt')  if HAVE_V8  else None
v11_model = YOLO('best_v11.pt') if HAVE_V11 else None

if v8_model:  print('Loaded AFTER v8:  best_v8.pt  (YOLOv8m fine-tuned, 9.5 h cap)')
if v11_model: print('Loaded AFTER v11: best_v11.pt (YOLOv11m fine-tuned, 8.0 h cap)')

## 3.5  Sample shelf images — ten public retail-shelf scenes

Ten public images covering a mix of scenes (grocery, beverage, drugstore, snack aisle, etc.)
are downloaded from public CDNs — no auth required. To swap in your own shelf photos, run
the optional upload cell that follows.

In [ ]:
import urllib.request
os.makedirs('samples', exist_ok=True)

# Ten publicly-hosted shelf images. All Ultralytics-bundled or public Unsplash assets;
# none require auth. If a URL ever 404s, just drop it - the loop below skips silently.
SAMPLE_URLS = [
    'https://raw.githubusercontent.com/ultralytics/assets/main/im/grocery.jpg',
    'https://images.unsplash.com/photo-1604719312566-8912e9227c6a?w=1600',  # fridge drinks
    'https://images.unsplash.com/photo-1578916171728-46686eac8d58?w=1600',  # bakery shelf
    'https://images.unsplash.com/photo-1583912086096-8c60d75a53f9?w=1600',  # canned goods
    'https://images.unsplash.com/photo-1542838132-92c53300491e?w=1600',     # produce row
    'https://images.unsplash.com/photo-1556767576-cf0a4a80a4c5?w=1600',     # snack aisle
    'https://images.unsplash.com/photo-1604719312566-8912e9227c6a?w=1600',  # cooler
    'https://images.unsplash.com/photo-1488459716781-31db52582fe9?w=1600',  # cereal aisle
    'https://images.unsplash.com/photo-1542838132-92c53300491e?w=1600',     # market produce
    'https://images.unsplash.com/photo-1591189824344-9c5c0a1e0f31?w=1600',  # liquor shelf
]

sample_paths = []
for i, url in enumerate(SAMPLE_URLS):
    p = f'samples/shelf_{i:02d}.jpg'
    try:
        urllib.request.urlretrieve(url, p)
        sample_paths.append(p)
    except Exception as e:
        print(f'Skipped {url}: {e}')

print(f'Fetched {len(sample_paths)} / {len(SAMPLE_URLS)} sample images.')

In [ ]:
# OPTIONAL - upload your own shelf images and append them to sample_paths.
# Uncomment to use.
#
# from google.colab import files
# extra = files.upload()
# sample_paths += [k for k in extra if k.lower().endswith(('.jpg', '.jpeg', '.png'))]

sample_paths = sample_paths[:N_SAMPLES]
assert sample_paths, 'No sample images! Re-run the previous cell.'
print(f'Using {len(sample_paths)} samples.')
sample_paths

## 3.6  Before vs. After — side by side

For each image we draw an N-up panel with one column per loaded model: **BEFORE (COCO)**
on the left, **AFTER v8** in the middle, **AFTER v11** on the right (whichever variants you
uploaded). Detection count in each title.

This is the checkpoint's punchline — the fine-tuned models find the hundreds of
densely-packed products the COCO model misses entirely, and v8 vs. v11 are visually
comparable at a glance.

In [ ]:
def annotated(model, path):
    """Run inference and return (RGB annotated image, num_detections)."""
    r = model.predict(path, conf=CONF, imgsz=IMGSZ, verbose=False)[0]
    bgr = r.plot()                 # numpy array, BGR with boxes drawn
    rgb = bgr[:, :, ::-1]          # to RGB for matplotlib
    return rgb, len(r.boxes)

# Build the column list dynamically based on which variants were uploaded.
columns = [('BEFORE - COCO yolov8m', before_model)]
if v8_model:  columns.append(('AFTER v8 - fine-tuned',  v8_model))
if v11_model: columns.append(('AFTER v11 - fine-tuned', v11_model))

os.makedirs('panels', exist_ok=True)
panel_paths = []

for i, path in enumerate(sample_paths):
    fig, axes = plt.subplots(1, len(columns), figsize=(8 * len(columns), 9))
    if len(columns) == 1:
        axes = [axes]

    counts = []
    for ax, (label, mdl) in zip(axes, columns):
        img, n = annotated(mdl, path)
        ax.imshow(img)
        ax.set_title(f'{label}  .  {n} detections', fontsize=14)
        ax.axis('off')
        counts.append(n)

    fig.suptitle(os.path.basename(path), fontsize=12)
    fig.tight_layout()
    out = f'panels/before_after_{i:02d}.png'
    fig.savefig(out, dpi=120, bbox_inches='tight')
    panel_paths.append(out)
    plt.show()

    summary = '  '.join(f'{lbl.split(" - ")[0]}={n}' for (lbl, _), n in zip(columns, counts))
    print(f'{os.path.basename(path)}:  {summary}')

## 3.7  Metrics summary (from the real GCP runs)

The **honest** metrics from the L4 training run(s), read straight from the `metrics_*.json`
files you uploaded in §3.3 — not from this short demo.

In [ ]:
def show_metrics(label, path):
    if not os.path.exists(path):
        print(f'{label}: not uploaded - skipping.')
        return
    metrics = json.load(open(path))
    targets = {'mAP_50': 0.70, 'precision': 0.75, 'recall': 0.70}
    print(f'\n=== {label} ===')
    print(f"{'Metric':<14}{'Value':<10}{'Target':<10}{'Pass?'}")
    print('-' * 44)
    for k, lab in [('mAP_50','mAP@0.5'), ('mAP_50_95','mAP@0.5:0.95'),
                   ('precision','Precision'), ('recall','Recall')]:
        v = metrics.get(k)
        t = targets.get(k)
        ok = '' if t is None else ('PASS' if v >= t else 'below')
        print(f"{lab:<14}{v:<10}{(t if t else '-'):<10}{ok}")
    print(f"Best epoch: {metrics.get('epoch')}")

show_metrics('YOLOv8m (9.5 h cap)',  'metrics_v8.json')
show_metrics('YOLOv11m (8.0 h cap)', 'metrics_v11.json')

In [ ]:
# Training curves (loss / mAP over epochs) from each GCP run.
from IPython.display import Image, display
for label, path in [('YOLOv8m', 'results_v8.png'), ('YOLOv11m', 'results_v11.png')]:
    if os.path.exists(path):
        print(f'--- {label} training curves ---')
        display(Image(path))

## 3.8  Save the panels (optional)

The rendered before/after panels are already on disk under `panels/`. Right-click any inline
figure to save, or run the cell below to bundle them into a zip you can download via the Colab
file browser.

In [ ]:
import shutil
if panel_paths:
    shutil.make_archive('before_after_panels', 'zip', 'panels')
    print('Wrote before_after_panels.zip - download from the Colab Files panel.')
else:
    print('No panels were rendered.')